# ELECTROVOLT — 05. Evaluación Final, Metas SMART y XAI (SHAP)
**Proyecto Integrador · Maestría en IA · UEES**

**Autores:** Jordan Vera Coello & Jorge Estrella Frere

En este notebook se realiza la validación formal de los criterios de aceptación y la explicabilidad física:
1. Carga del paquete oficial desde `models/best_model.pkl`.
2. Evaluación rigurosa de las **Metas SMART** (`Sí` / `No`).
3. Matriz de confusión final e informe de clasificación.
4. IA Explicable con SHAP:
   - Importancia global de las 6 señales trifásicas.
   - Mapa de calor de atribución por clase de falla.
   - Explicación local de una predicción concreta para el operador.
5. Prueba de la función modular `predecir_falla(...)` simulando la interfaz web.


In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.model import cargar_paquete_modelo, predecir_falla
from src.data_processing import cargar_datos, mapear_clases, dividir_datos
from src.evaluate import (
    generar_reporte_metricas,
    graficar_matriz_confusion,
    graficar_explicabilidad_shap_global,
    graficar_shap_heatmap,
    graficar_shap_local,
)
from src.utils import COLUMNAS, ORDEN_CLASES, SEMILLA

sns.set_theme(style="whitegrid")
print("[OK] Módulos de evaluación y XAI listos.")


## 1. Carga del modelo oficial y datos de prueba


In [ ]:
paquete = cargar_paquete_modelo()
print(f"Modelo cargado: {paquete['descripcion']}")
print(f"Clases reconocidas: {paquete['clases']}")

df = mapear_clases(cargar_datos(), modo="5_clases")
_, X_pru, _, y_pru, _, _ = dividir_datos(df, test_size=0.20, random_state=SEMILLA)
rf = paquete["modelo"]
pred = rf.predict(X_pru)


## 2. Auditoría contra las Metas SMART del proyecto
Comprobamos cada uno de los umbrales exigidos en `02-project-context-and-goals.md`.


In [ ]:
reporte = generar_reporte_metricas(
    y_real=y_pru,
    y_pred=pred,
    modelo=rf,
    X_uno=X_pru.iloc[[0]],
    X_pru=X_pru,
    tiempo_entrenamiento=0.89,
    clases=ORDEN_CLASES
)

print("=== AUDITORÍA FORMAL DE METAS SMART ===")
reporte["tabla_smart"]


## 3. Matriz de confusión y reporte de clasificación


In [ ]:
print("=== CLASSIFICATION REPORT ===")
print(reporte["reporte_clasificacion"])

fig_cm = graficar_matriz_confusion(y_pru, pred, clases=ORDEN_CLASES)
plt.show()


## 4. IA Explicable con SHAP
Explicamos cómo pondera el árbol las magnitudes de corriente y tensión.


In [ ]:
fig_shap = graficar_explicabilidad_shap_global(rf, X_pru, clases=ORDEN_CLASES, columnas=COLUMNAS)
plt.show()

fig_heat = graficar_shap_heatmap(rf, X_pru, clases=ORDEN_CLASES, columnas=COLUMNAS)
plt.show()


## 5. Explicación Local de una Muestra Concreta para el Operador
Generamos el gráfico de barras individual para una muestra real de falla trifásica.


In [ ]:
idx_tri = y_pru[y_pru == "Trifásica"].index[0]
fila_tri = X_pru.loc[[idx_tri]]

fig_local = graficar_shap_local(rf, fila_tri, clases=ORDEN_CLASES, columnas=COLUMNAS)
plt.show()


## 6. Prueba en vivo de la función `predecir_falla(...)`
Ejecutamos el contrato de inferencia sobre ejemplos de cada una de las clases de falla.


In [ ]:
ejemplos = pd.read_csv(raiz / "data" / "processed" / "ejemplos_prueba.csv")

for _, r in ejemplos.groupby("clase").head(1).iterrows():
    res = predecir_falla(r["Va"], r["Vb"], r["Vc"], r["Ia"], r["Ib"], r["Ic"], paquete)
    print(f"Real: {r['clase']:<10} -> Diagnóstico: {res['diagnostico']:<10} (Confianza: {res['confianza']:.1%})")

print("\nMensaje operativo para el operador ante falla trifásica:")
print(f"'{res['mensaje_operador']}'")
